# Embeddings & Semantic Search — live demo

Companion notebook for the 45-minute session plan (`session-plan.html`).

**Run cell 0 before you start screen-sharing** so the model is cached and nothing
downloads in front of an audience. Then clear all outputs, so results appear live.

CPU only — no GPU needed anywhere in this notebook.

## Cell 0 — setup (run before the session)

In [1]:
from sentence_transformers import SentenceTransformer, util
import pandas as pd

model = SentenceTransformer("all-MiniLM-L6-v2")  # ~80 MB, cached after first run
print("ready")


ready


## Run 1 — look at the actual numbers  *(2 min)*

> "That's the word 'king', as far as the machine is concerned. 384 numbers, and I'm
> showing you the first eight. This is the whole thing. There is no meaning hiding
> anywhere else — no dictionary, no rules file. Just this list."

In [6]:
v = model.encode("king")

print(v.shape)            # (384,)
print(v[:8].round(3))     # [-0.06  0.051 -0.07  0.08 -0.047  0.001  0.079 -0.013]

(384,)
[-0.06   0.051 -0.07   0.08  -0.047  0.001  0.079 -0.013]


## Run 2 — similar things score high  *(3 min)*

**Ask the chat before running:** sentence 0 is about pizza, sentence 3 is about Python.
High score or low?

Then point at **row 0, column 1** — `0.54` — and **row 0, column 5** — `0.04`.
Neither pair shares a single content word.

In [ ]:
sentences = [
    "I love eating pizza",
    "Pasta is my favourite food",
    "This restaurant serves great biryani",
    "Python is a programming language",
    "I write code every day",
    "Debugging takes most of my time",
]

emb = model.encode(sentences)
sim = util.cos_sim(emb, emb)

pd.DataFrame(sim.numpy().round(2),
             index=[s for s in sentences],
             columns=range(6))

Verified output:

```
                               0     1     2     3     4     5
I love eating pizza         1.00  0.54  0.24  0.10  0.19  0.04
Pasta is my favourite food  0.54  1.00  0.32  0.14  0.18  0.01
This restaurant serves gre  0.24  0.32  1.00 -0.09 -0.02 -0.02
Python is a programming la  0.10  0.14 -0.09  1.00  0.30  0.10
I write code every day      0.19  0.18 -0.02  0.30  1.00  0.27
Debugging takes most of my  0.04  0.01 -0.02  0.10  0.27  1.00
```

**Honest aside worth making:** biryani only scores `0.24` against pizza — lower than you'd
expect for two foods. The sentence is framed around a *restaurant*, not around *eating*,
and that shifts its vector. Real embeddings are messier than clean slides suggest, and
saying so is worth more than pretending the matrix is perfect.

## Run 3 — fix the search box from minute one  *(5 min)*

The emotional peak. Keyword search matches only on `account`, so it confidently returns
*"How do I delete my account permanently?"* — the user asked for help getting **in**, and
it offered to destroy their account.

Semantic search puts password-reset first, having never seen the word "password" in the query.

In [8]:
faq = [
    "How do I reset a forgotten password?",
    "The app crashes when I upload a photo.",
    "What is your refund and return policy?",
    "How long does standard shipping take?",
    "Can I change the email on my profile?",
    "My card was declined at checkout.",
    "How do I delete my account permanently?",
    "Is there a student discount available?",
]
faq_emb = model.encode(faq, convert_to_tensor=True)

query = "I can't log in to my account"

# ---------- the old way ----------
STOP = {"i", "my", "to", "the", "a", "is", "in", "do", "how", "can", "can't"}
clean = lambda s: {w.strip("?.,'") for w in s.lower().split()} - STOP

print("KEYWORD SEARCH")
for f in faq:
    if clean(query) & clean(f):
        print("   ", f)

# ---------- the new way ----------
q_emb = model.encode(query, convert_to_tensor=True)
print("\nSEMANTIC SEARCH")
for hit in util.semantic_search(q_emb, faq_emb, top_k=3)[0]:
    print(f"   {hit['score']:.3f}   {faq[hit['corpus_id']]}")

KEYWORD SEARCH
    How do I delete my account permanently?

SEMANTIC SEARCH
   0.502   How do I reset a forgotten password?
   0.459   How do I delete my account permanently?
   0.241   The app crashes when I upload a photo.


Verified output:

```
KEYWORD SEARCH
    How do I delete my account permanently?

SEMANTIC SEARCH
    0.502   How do I reset a forgotten password?
    0.459   How do I delete my account permanently?
    0.241   The app crashes when I upload a photo.
```

### Run 3b — the honest follow-up  *(optional, ~1 min, first thing to cut)*

Second place was `0.459` against a winner of `0.502` — uncomfortably close. Change the
phrasing slightly and **the wrong answer wins outright**. This is measured, not
hypothetical, and it is the cleanest possible motivation for rerankers.

> "Retrieval is not a solved problem. This is why production systems don't stop at cosine
> similarity — they take the top 20 and pass them to a slower, smarter *reranker* that
> reads each candidate against the query properly."

In [9]:
for q in ["I can't log in to my account", "I'm locked out of my account"]:
    hits = util.semantic_search(model.encode(q, convert_to_tensor=True),
                               faq_emb, top_k=2)[0]
    print(f"query: {q!r}")
    for h in hits:
        print(f"   {h['score']:.3f}   {faq[h['corpus_id']]}")
    print()

query: "I can't log in to my account"
   0.502   How do I reset a forgotten password?
   0.459   How do I delete my account permanently?

query: "I'm locked out of my account"
   0.636   How do I delete my account permanently?
   0.539   How do I reset a forgotten password?



Verified output — note the second query ranks the *wrong* article first:

```
query: "I can't log in to my account"
   0.502   How do I reset a forgotten password?
   0.459   How do I delete my account permanently?

query: "I'm locked out of my account"
   0.636   How do I delete my account permanently?
   0.539   How do I reset a forgotten password?
```

## Run 4 — now break it on purpose  *(3 min)*

Most demo sessions stop at the win. Ending on a limitation is what makes you read as a
practitioner rather than someone who followed a tutorial last night.

In [10]:
pairs = [
    ("The flight was on time",  "The flight was delayed"),    # opposites
    ("I love this product",     "I hate this product"),       # opposites
    ("I love this product",     "This product is wonderful"),  # genuinely similar
]

for a, b in pairs:
    s = util.cos_sim(model.encode(a), model.encode(b)).item()
    print(f"{s:.3f}   {a!r}  vs  {b!r}")

0.800   'The flight was on time'  vs  'The flight was delayed'
0.698   'I love this product'  vs  'I hate this product'
0.829   'I love this product'  vs  'This product is wonderful'


Verified output:

```
0.800   'The flight was on time'  vs  'The flight was delayed'
0.698   'I love this product'     vs  'I hate this product'
0.829   'I love this product'     vs  'This product is wonderful'
```

**The line to land:** a pair of *opposites* scores `0.800`, sitting a hair under a
genuinely similar pair at `0.829`.

> "'On time' and 'delayed' mean opposite things and they're scoring 0.800 — basically tied
> with a pair that genuinely does match. Why? Because these vectors capture **what a
> sentence is about** far better than **which side of it you're on**. So: never build a
> sentiment classifier out of raw cosine similarity, and if your search has to respect a
> 'not', embeddings alone will not save you."
>
> "Every technique you learn has an edge like this. The engineers who get hired are the
> ones who know where the edge is."

---

## Verification record

Every output above was produced on:

| | |
| --- | --- |
| Model | `all-MiniLM-L6-v2` (384 dims) |
| Python | 3.12.10, CPU only |
| torch | 2.13.0+cpu |
| Platform | Windows 11 |

Scores from this model are deterministic, so re-running should reproduce these numbers
exactly. A different model — or a different `sentence-transformers` major version — will
shift them, so re-verify if you swap either.